## 1. Load Data & Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder

BASE = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series'

# Kaggle competition test set (no sales labels)
test_kaggle  = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'test_cleaned.csv'))

# Full historical training data — used to compute lag / rolling features
train_raw    = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'train_cleaned.csv'))

oil_df       = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'oil.csv'))
holidays_df  = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'holidays_events.csv'))
stores_df    = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'stores.csv'))

# Authoritative feature column order from splits
ref_cols = pd.read_csv(
    os.path.join(BASE, 'data', 'processed', 'splits', 'train_features.csv'),
    nrows=0
).columns.tolist()

for name, df in [('test_kaggle', test_kaggle), ('train_raw', train_raw),
                  ('oil', oil_df), ('holidays', holidays_df), ('stores', stores_df)]:
    print(f'{name:12s}: {df.shape}')
print(f'{"ref_cols":12s}: {len(ref_cols)} features')

test_kaggle : (28512, 5)
train_raw   : (3000888, 6)
oil         : (1218, 2)
holidays    : (350, 6)
stores      : (54, 5)
ref_cols    : 49 features


## 2. Feature Engineering

In [2]:
# --- Calendar features (identical logic to feature_temporal.ipynb) ---
X_kaggle = test_kaggle.copy()
X_kaggle['date'] = pd.to_datetime(X_kaggle['date'])

train_raw['date'] = pd.to_datetime(train_raw['date'])
train_raw['sales_log'] = np.log1p(train_raw['sales'])

dt = X_kaggle['date'].dt
X_kaggle['year']           = dt.year
X_kaggle['month']          = dt.month
X_kaggle['day']            = dt.day
X_kaggle['day_of_week']    = dt.dayofweek          # 0=Mon, 6=Sun
X_kaggle['week_of_year']   = dt.isocalendar().week.astype(int)
X_kaggle['quarter']        = dt.quarter
X_kaggle['is_weekend']     = (dt.dayofweek >= 5).astype(int)
X_kaggle['is_month_start'] = dt.is_month_start.astype(int)
X_kaggle['is_month_end']   = dt.is_month_end.astype(int)
X_kaggle['is_payday']      = ((dt.day == 15) | dt.is_month_end).astype(int)

# Merge store geographic info (needed for holiday matching AND as features)
X_kaggle = X_kaggle.merge(
    stores_df[['store_nbr', 'type', 'city', 'state', 'cluster']],
    on='store_nbr', how='left'
)

assert X_kaggle.shape[0] == len(test_kaggle), 'Store merge changed row count'
print(f'After calendar + store merge: {X_kaggle.shape}')

After calendar + store merge: (28512, 19)


In [3]:
# --- Lag features (from train_raw sales_log, minimum lag = 16 days safe horizon) ---
# Shift the reference date forward by lag so a left-merge finds the correct history row.
# Lags 1/2/3/7/14 will be NaN for dates where the look-back falls inside the test window.
for lag in [1, 2, 3, 7, 14, 28, 364]:
    lag_ref = train_raw[['date', 'store_nbr', 'family', 'sales_log']].copy()
    lag_ref['date'] = lag_ref['date'] + pd.Timedelta(days=lag)
    lag_ref = lag_ref.rename(columns={'sales_log': f'lag_{lag}'})
    X_kaggle = X_kaggle.merge(
        lag_ref[['date', 'store_nbr', 'family', f'lag_{lag}']],
        on=['date', 'store_nbr', 'family'], how='left'
    )

assert X_kaggle.shape[0] == len(test_kaggle), 'Lag merge changed row count'

lag_cols = [f'lag_{l}' for l in [1, 2, 3, 7, 14, 28, 364]]
print('Lag NaN counts (expected for short lags — test window is 16 days):')
for c in lag_cols:
    n = X_kaggle[c].isnull().sum()
    print(f'  {c:8s}: {n:>6d} ({n / len(X_kaggle) * 100:.1f}%)')

Lag NaN counts (expected for short lags — test window is 16 days):
  lag_1   :  26730 (93.8%)
  lag_2   :  24948 (87.5%)
  lag_3   :  23166 (81.2%)
  lag_7   :  16038 (56.2%)
  lag_14  :   3564 (12.5%)
  lag_28  :      0 (0.0%)
  lag_364 :      0 (0.0%)


In [4]:
# --- Rolling statistics (replicate training behavior: groupby shift(1) then flat rolling) ---
# Only the last 30 days of train are needed; the rolling window spans < 1 date in flat order.
cutoff = X_kaggle['date'].min() - pd.Timedelta(days=30)
train_recent = train_raw.loc[
    train_raw['date'] >= cutoff, ['date', 'store_nbr', 'family', 'sales_log']
]

forecast_combined = pd.concat([
    train_recent,
    X_kaggle[['date', 'store_nbr', 'family']].assign(sales_log=np.nan)
]).sort_values(['date', 'store_nbr', 'family']).reset_index(drop=True)

# GroupBy shift(1) within each (store, family) — then rolling on the flat sorted series
shifted = forecast_combined.groupby(['store_nbr', 'family'])['sales_log'].shift(1)

forecast_combined['rolling_mean_7']  = shifted.rolling(7,  min_periods=1).mean()
forecast_combined['rolling_mean_14'] = shifted.rolling(14, min_periods=1).mean()
forecast_combined['rolling_mean_28'] = shifted.rolling(28, min_periods=1).mean()
forecast_combined['rolling_std_7']   = shifted.rolling(7,  min_periods=2).std()

rolling_cols = ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
kaggle_mask = forecast_combined['date'] >= pd.Timestamp('2017-08-16')
kaggle_rolling = forecast_combined.loc[
    kaggle_mask, ['date', 'store_nbr', 'family'] + rolling_cols
].reset_index(drop=True)

X_kaggle = X_kaggle.merge(kaggle_rolling, on=['date', 'store_nbr', 'family'], how='left')
del forecast_combined, shifted, kaggle_rolling

assert X_kaggle.shape[0] == len(test_kaggle), 'Rolling merge changed row count'
print('Rolling NaN counts (NaN expected for dates > first test day):')
for c in rolling_cols:
    print(f'  {c:20s}: {X_kaggle[c].isnull().sum()}')

Rolling NaN counts (NaN expected for dates > first test day):
  rolling_mean_7      : 26724
  rolling_mean_14     : 26717
  rolling_mean_28     : 26703
  rolling_std_7       : 26725


In [5]:
# --- Holiday features (replicate feature_holiday.ipynb logic) ---
holidays = holidays_df.copy()
holidays['date'] = pd.to_datetime(holidays['date'])

# transferred=True means the original date is void; Transfer rows (transferred=False) are the observed dates
valid_holidays = holidays[holidays['transferred'] == False].copy()

type_mapping = {
    'Holiday': 1, 'Event': 2, 'Additional': 3,
    'Transfer': 4, 'Bridge': 5, 'Work Day': 0
}
valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0).astype(int)
valid_holidays['is_transferred']       = valid_holidays['transferred'].astype(int)
valid_holidays['is_carnaval']          = (
    valid_holidays['description'].str.contains('Carnaval', case=False, na=False).astype(int)
)

# Initialise all holiday columns to 0
for col in ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
            'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']:
    X_kaggle[col] = 0

scope_col = {'National': 'is_national_holiday',
             'Regional': 'is_regional_holiday',
             'Local':    'is_local_holiday'}

for _, row in valid_holidays.iterrows():
    h_date, h_locale, h_name = row['date'], row['locale'], row['locale_name']
    if h_locale == 'National':
        mask = X_kaggle['date'] == h_date
    elif h_locale == 'Regional':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['state'] == h_name)
    elif h_locale == 'Local':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['city'] == h_name)
    else:
        continue
    if not mask.any():
        continue
    X_kaggle.loc[mask, scope_col[h_locale]]      = 1
    X_kaggle.loc[mask, 'is_transferred_holiday'] = row['is_transferred']
    X_kaggle.loc[mask, 'holiday_type']           = row['holiday_type_encoded']
    X_kaggle.loc[mask, 'is_carnaval_feature']    = row['is_carnaval']

# Halo effect: days to next / days after last valid holiday
unique_hol_dates = valid_holidays['date'].drop_duplicates().sort_values().values

def _days_to_next(d):
    future = unique_hol_dates[unique_hol_dates > d]
    return int((future[0] - d) / np.timedelta64(1, 'D')) if len(future) > 0 else -1

def _days_after_last(d):
    past = unique_hol_dates[unique_hol_dates < d]
    return int((d - past[-1]) / np.timedelta64(1, 'D')) if len(past) > 0 else -1

uq_test_dates = X_kaggle[['date']].drop_duplicates().copy()
uq_test_dates['days_to_next_holiday']    = uq_test_dates['date'].apply(_days_to_next)
uq_test_dates['days_after_last_holiday'] = uq_test_dates['date'].apply(_days_after_last)
X_kaggle = X_kaggle.merge(uq_test_dates, on='date', how='left')

# Earthquake period was Apr 16 – May 16 2016; test dates are Aug 2017 → always 0
X_kaggle['is_earthquake_period'] = 0

assert X_kaggle.shape[0] == len(test_kaggle), 'Holiday merge changed row count'
print(f'National holidays hit: {X_kaggle["is_national_holiday"].sum()}')
print(f'Regional holidays hit: {X_kaggle["is_regional_holiday"].sum()}')
print(f'Local holidays hit   : {X_kaggle["is_local_holiday"].sum()}')

National holidays hit: 0
Regional holidays hit: 0
Local holidays hit   : 66


In [6]:
# --- Oil price features (replicate feature_oil_and_store.ipynb logic) ---
oil = oil_df.copy()
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
oil['date'] = pd.to_datetime(oil['date'])
oil = oil[['date', 'oil_price']].sort_values('date').reset_index(drop=True)

# Extend to cover full test period then forward-fill missing days (weekends, holidays)
max_date = max(oil['date'].max(), X_kaggle['date'].max())
full_range = pd.DataFrame({'date': pd.date_range(oil['date'].min(), max_date, freq='D')})
oil = full_range.merge(oil, on='date', how='left')
oil['oil_price'] = oil['oil_price'].ffill().bfill()

oil['oil_price_lag_7']           = oil['oil_price'].shift(7)
oil['oil_price_lag_14']          = oil['oil_price'].shift(14)
oil['oil_price_rolling_mean_28'] = oil['oil_price'].shift(1).rolling(28, min_periods=7).mean()
oil['oil_price_change_pct']      = oil['oil_price'].pct_change(periods=7)

oil_feat_cols = ['date', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14',
                 'oil_price_rolling_mean_28', 'oil_price_change_pct']
X_kaggle = X_kaggle.merge(oil[oil_feat_cols], on='date', how='left')

assert X_kaggle.shape[0] == len(test_kaggle), 'Oil merge changed row count'
print(f'Oil NaN total: {X_kaggle[oil_feat_cols[1:]].isnull().sum().sum()}')
print(f'Oil price range: {X_kaggle["oil_price"].min():.2f} – {X_kaggle["oil_price"].max():.2f}')

Oil NaN total: 0
Oil price range: 45.96 – 48.59


In [7]:
# --- Store encoding + target encoding + column assertion ---

# Store type encoding — alphabetical (A=0, B=1, C=2, D=3, E=4), same as LabelEncoder on type
type_map = {t: i for i, t in enumerate(sorted(stores_df['type'].unique()))}
X_kaggle['store_type_enc']     = X_kaggle['type'].map(type_map)
X_kaggle['store_type_encoded'] = X_kaggle['type'].map(type_map)

# Frequency encoding — proportion of stores per city / state (identical to training FE)
city_freq_map  = stores_df['city'].value_counts(normalize=True).to_dict()
state_freq_map = stores_df['state'].value_counts(normalize=True).to_dict()
X_kaggle['city_freq']  = X_kaggle['city'].map(city_freq_map)
X_kaggle['state_freq'] = X_kaggle['state'].map(state_freq_map)

# Store-family composite key
X_kaggle['store_family'] = X_kaggle['store_nbr'].astype(str) + '_' + X_kaggle['family']

# Target encoding — fit TargetEncoder on full train, same as test-set encoding in training pipeline
train_raw['store_family'] = train_raw['store_nbr'].astype(str) + '_' + train_raw['family']
te_enc = TargetEncoder(cols=['store_family'], smoothing=10)
te_enc.fit(train_raw[['store_family']], train_raw['sales'])
X_kaggle['store_family_te'] = te_enc.transform(X_kaggle[['store_family']])['store_family']

# Convert date back to string format matching training CSVs (YYYY-MM-DD)
X_kaggle['date'] = X_kaggle['date'].dt.strftime('%Y-%m-%d')

# Drop id — save it for the submission DataFrame
kaggle_ids = test_kaggle['id'].reset_index(drop=True)
X_kaggle = X_kaggle.drop(columns=['id'], errors='ignore')

# Assert exact column match against train_features.csv
missing_cols = set(ref_cols) - set(X_kaggle.columns)
extra_cols   = set(X_kaggle.columns) - set(ref_cols)
assert not missing_cols, f'Missing columns: {missing_cols}'
assert not extra_cols,   f'Extra columns: {extra_cols}'

X_kaggle = X_kaggle[ref_cols]  # enforce exact column order
assert X_kaggle.shape == (len(test_kaggle), len(ref_cols)), 'Final shape mismatch'

print(f'X_kaggle: {X_kaggle.shape}')
nan_counts = X_kaggle.isnull().sum()
nan_nonzero = nan_counts[nan_counts > 0]
if len(nan_nonzero):
    print(f'Columns with NaN (expected for short lags):\n{nan_nonzero}')
else:
    print('No missing values')

X_kaggle: (28512, 49)
Columns with NaN (expected for short lags):
lag_1              26730
lag_2              24948
lag_3              23166
lag_7              16038
lag_14              3564
rolling_mean_7     26724
rolling_mean_14    26717
rolling_mean_28    26703
rolling_std_7      26725
dtype: int64


## 3. Encode Categorical Features

In [8]:
# Refit LabelEncoders on train + val + test splits — identical to training notebook logic.
# X_kaggle is also included so that new Kaggle dates (2017-08-16 to 2017-08-31) are in-vocabulary.
# Because LabelEncoder sorts alphabetically and test dates come after all training dates,
# adding them does NOT change existing integer mappings.

splits_dir = os.path.join(BASE, 'data', 'processed', 'splits')
object_cols = X_kaggle.select_dtypes(include=['object']).columns.tolist()

# Load only object columns from each split to minimise memory
X_train_split = pd.read_csv(
    os.path.join(splits_dir, 'train_features.csv'),
    usecols=lambda c: c in object_cols
)
X_val_split   = pd.read_csv(
    os.path.join(splits_dir, 'val_features.csv'),
    usecols=lambda c: c in object_cols
)
X_test_split  = pd.read_csv(
    os.path.join(splits_dir, 'test_features.csv'),
    usecols=lambda c: c in object_cols
)

label_encoders = {}
for col in object_cols:
    le = LabelEncoder()
    parts = [X_train_split[col], X_val_split[col]]
    if col in X_test_split.columns:
        parts.append(X_test_split[col])
    parts.append(X_kaggle[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)
    X_kaggle[col] = le.transform(X_kaggle[col].astype(str)).astype(np.int32)
    label_encoders[col] = le

del X_train_split, X_val_split, X_test_split

# Verify no object columns remain
remaining_obj = X_kaggle.select_dtypes(include=['object']).columns.tolist()
assert not remaining_obj, f'Still has object columns: {remaining_obj}'

print(f'Encoded columns: {object_cols}')

Encoded columns: ['date', 'family', 'type', 'city', 'state', 'store_family']


## 4. Load Models & Predict

In [9]:
lgb_best_model = joblib.load(os.path.join(BASE, 'notebooks', '08_forecasting', 'lgb_best_model.pkl'))
xgb_best_model = joblib.load(os.path.join(BASE, 'notebooks', '08_forecasting', 'xgb_best_model.pkl'))

# Source: Model_ensemble.ipynb — LGB5-XGB5 achieved lowest Test RMSLE (0.370576)
best_lgb_w = 0.3
best_xgb_w = 0.7

lgb_pred = lgb_best_model.predict(X_kaggle)
xgb_pred = xgb_best_model.predict(X_kaggle)

# Weighted blend in log space, then inverse log1p; clip negatives to 0
final_pred_log = best_lgb_w * lgb_pred + best_xgb_w * xgb_pred
final_pred     = np.maximum(np.expm1(final_pred_log), 0)

print(f'LGB weight : {best_lgb_w}   XGB weight : {best_xgb_w}')
print(f'Predictions — min: {final_pred.min():.4f}  max: {final_pred.max():.2f}  mean: {final_pred.mean():.2f}')

[LightGBM] [Warning] Unknown parameter: tree_method
LGB weight : 0.3   XGB weight : 0.7
Predictions — min: 0.4252  max: 6658.21  mean: 1019.91


## 5. Submission

In [10]:
submission = pd.DataFrame({'id': kaggle_ids, 'sales': final_pred})

# Validation
assert len(submission) == len(test_kaggle),          'Row count mismatch'
assert submission['sales'].isnull().sum() == 0,      'NaN in sales'
assert (submission['sales'] < 0).sum() == 0,         'Negative sales'

output_path = os.path.join(BASE, 'notebooks', '08_ensemble', 'submission.csv')
submission.to_csv(output_path, index=False)

print(f'Shape  : {submission.shape}')
print(f'Sales  — min: {submission["sales"].min():.4f}  max: {submission["sales"].max():.2f}  mean: {submission["sales"].mean():.2f}')
print(submission.head())

Shape  : (28512, 2)
Sales  — min: 0.4252  max: 6658.21  mean: 1019.91
        id      sales
0  3000888   2.883112
1  3000889   1.099955
2  3000890   5.686706
3  3000891  15.661010
4  3000892   0.854783
